# Data Exploration + Preprocessing: <YOUR NAME>

**How to use this template**
1. Copy this file to `eda/eda_<yourname>.ipynb` and work only in your copy (on your own branch).
2. Run the cells group by group. Under each group, write your findings in the **Findings** cell.
3. At the end, fill in the decision table (one row per feature) and save it. This table is what we compare in the team meeting.
4. Upload your notebook and your CSV to the exploration branch (see README). Deadline: 11 October, 10:00 (German time).

**Rules**
- Use only the training split (`load_train()`, 80% of the data). Never look at the test split.
- Do not change `src/` without telling the team; everyone relies on it.

**Guiding questions for every group**
- Distribution: skewed? many zeros? strange values (negative, extreme, placeholder like 0 / "Unknown")?
- Missing values: how many, and how should they be handled?
- Relation to churn: does the churn rate change across values?
- Redundancy: are some columns highly correlated or derived from each other?
- Preprocessing: keep or drop? impute how? transform (log, clip, binarise)? encode how?
- Feature engineering: which new features could be useful?

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import load_train
from src.eda_utils import (FEATURE_GROUPS, overview, churn_signal, churn_rate_by,
                           plot_column, corr_heatmap, summary_skeleton)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

train = load_train()
print(train.shape)
print("churn rate:", round(train["churn"].mean(), 4))

## 0. Overview of all features

In [ ]:
all_cols = [c for cols in FEATURE_GROUPS.values() for c in cols]
overview(train, all_cols)

## 1. Account & billing

Columns: `MonthlyRevenue`, `TotalRecurringCharge`, `OverageMinutes`, `MonthsInService`, `CurrentEquipmentDays`, `Handsets`, `HandsetModels`, `UniqueSubs`, `ActiveSubs`, `AdjustmentsToCreditRating`

In [ ]:
cols = FEATURE_GROUPS["account_billing"]
display(overview(train, cols))
display(churn_signal(train, cols))

In [ ]:
for c in cols:
    plot_column(train, c)

In [ ]:
corr = corr_heatmap(train, cols)

**Findings (Account & billing):**

- 
- 
- 

## 2. Service usage

Columns: `MonthlyMinutes`, `DirectorAssistedCalls`, `RoamingCalls`, `ThreewayCalls`, `ReceivedCalls`, `OutboundCalls`, `InboundCalls`, `PeakCallsInOut`, `OffPeakCallsInOut`, `CallForwardingCalls`, `CallWaitingCalls`

In [ ]:
cols = FEATURE_GROUPS["usage"]
display(overview(train, cols))
display(churn_signal(train, cols))

In [ ]:
for c in cols:
    plot_column(train, c)

In [ ]:
corr = corr_heatmap(train, cols)

**Findings (Service usage):**

- 
- 
- 

## 3. Usage change & retention (check for leakage!)

Columns: `PercChangeMinutes`, `PercChangeRevenues`, `RetentionCalls`, `RetentionOffersAccepted`, `MadeCallToRetentionTeam`, `ReferralsMadeBySubscriber`, `NewCellphoneUser`, `NotNewCellphoneUser`

In [ ]:
cols = FEATURE_GROUPS["usage_change_retention"]
display(overview(train, cols))
display(churn_signal(train, cols))

In [ ]:
for c in cols:
    plot_column(train, c)

In [ ]:
corr = corr_heatmap(train, cols)

**Findings (Usage change & retention (check for leakage!)):**

- 
- 
- 

## 4. Service quality

Columns: `DroppedCalls`, `BlockedCalls`, `UnansweredCalls`, `DroppedBlockedCalls`, `CustomerCareCalls`

In [ ]:
cols = FEATURE_GROUPS["service_quality"]
display(overview(train, cols))
display(churn_signal(train, cols))

In [ ]:
for c in cols:
    plot_column(train, c)

In [ ]:
corr = corr_heatmap(train, cols)

**Findings (Service quality):**

- 
- 
- 

## 5. Demographics & region

Columns: `AgeHH1`, `AgeHH2`, `ChildrenInHH`, `IncomeGroup`, `Occupation`, `MaritalStatus`, `Homeownership`, `CreditRating`, `PrizmCode`, `ServiceArea`

In [ ]:
cols = FEATURE_GROUPS["demographics_region"]
display(overview(train, cols))
display(churn_signal(train, cols))

In [ ]:
for c in cols:
    plot_column(train, c)

In [ ]:
corr = corr_heatmap(train, cols)

**Findings (Demographics & region):**

- 
- 
- 

## 6. Handset & lifestyle

Columns: `HandsetPrice`, `HandsetRefurbished`, `HandsetWebCapable`, `TruckOwner`, `RVOwner`, `OwnsMotorcycle`, `OwnsComputer`, `HasCreditCard`, `BuysViaMailOrder`, `RespondsToMailOffers`, `OptOutMailings`, `NonUSTravel`

In [ ]:
cols = FEATURE_GROUPS["handset_lifestyle"]
display(overview(train, cols))
display(churn_signal(train, cols))

In [ ]:
for c in cols:
    plot_column(train, c)

**Findings (Handset & lifestyle):**

- 
- 
- 

## 7. Cross-group checks

Checks that look at several columns together. Run them and note what they mean for preprocessing.

### 7.1 Redundant columns
Are some columns (almost) derived from others? If yes, we may only need one of them.

In [ ]:
dbc = train["DroppedCalls"] + train["BlockedCalls"]
print("DroppedBlockedCalls == DroppedCalls + BlockedCalls:",
      round(((train["DroppedBlockedCalls"] - dbc).abs() < 0.11).mean(), 3))
print(pd.crosstab(train["NewCellphoneUser"], train["NotNewCellphoneUser"]))
print(pd.crosstab(train["Handsets"].clip(upper=5), train["HandsetModels"].clip(upper=5)))
print(pd.crosstab(train["UniqueSubs"].clip(upper=5), train["ActiveSubs"].clip(upper=5)))

### 7.2 Are placeholder values informative?
If customers with an "unknown" value churn more or less than others, a missing-indicator feature is useful.

In [ ]:
from src.eda_utils import PLACEHOLDERS

rows = []
for c, v in PLACEHOLDERS.items():
    is_ph = train[c] == v
    rows.append({"column": c, "placeholder": v, "share": round(is_ph.mean(), 3),
                 "churn_if_placeholder": round(train.loc[is_ph, "churn"].mean(), 3),
                 "churn_otherwise": round(train.loc[~is_ph, "churn"].mean(), 3)})
pd.DataFrame(rows).set_index("column")

### 7.3 Churn by tenure, month by month
Is there a point in time where churn suddenly changes (e.g. contract expiry)?

In [ ]:
rate = train.groupby("MonthsInService")["churn"].agg(["size", "mean"])
rate = rate[rate["size"] >= 100]
ax = rate["mean"].plot(marker="o", figsize=(10, 3), title="churn rate by MonthsInService (months with >= 100 customers)")
ax.axhline(train["churn"].mean(), ls="--", c="grey");

### 7.4 Tenure × spending segments
These segments will be used later for the error analysis (which groups contain the most missed churners).

In [ ]:
tenure_grp = pd.cut(train["MonthsInService"], [0, 10, 12, 24, 1000], labels=["6-10m", "11-12m", "13-24m", ">24m"])
spend_grp = pd.qcut(train["MonthlyRevenue"], 3, labels=["low", "mid", "high"])
display(pd.crosstab(tenure_grp, spend_grp))                                          # size
display(pd.crosstab(tenure_grp, spend_grp, values=train["churn"], aggfunc="mean").round(3))  # churn rate

**Findings (cross-group):**

- 
- 
- 

## 8. Your own extra analysis (optional)

Anything else you find interesting, e.g. churn by `MonthsInService` month by month, interactions between groups, ideas for new features (try them here and check their churn rate / AUC).

## 9. Decision table

Fill in one row per feature. Keep entries short. Suggested values:
- `keep_drop`: keep / drop / unsure
- `missing_handling`: none / median / mode / placeholder→NaN + indicator / ...
- `transformation`: none / log1p / clip at p99 / clip at 0 / binarise / ...
- `encoding`: numeric / binary 0-1 / ordinal / one-hot / target encoding / ...

Example below; overwrite and add your own.

In [ ]:
summary = summary_skeleton()

summary.loc["MonthlyRevenue", ["issues", "relation_to_churn", "keep_drop", "missing_handling", "transformation", "encoding"]] = [
    "0.3% missing, a few negative values, right-skewed", "weak (AUC 0.49)", "keep", "median", "clip at 0, log1p", "numeric"]
# summary.loc["<column>", "feature_ideas"] = "..."

summary

In [ ]:
NAME = "yourname"  # <- change
summary.to_csv(f"summaries/eda_summary_{NAME}.csv")